In [ ]:
import h5py
import numpy as np
from pathlib import Path

# Paths relative to repo root
LOGS_ROOT = Path("../../logs/wicompass")
KNN_ROOT = LOGS_ROOT / "knn_coverage"
TOKEN_ROOT = LOGS_ROOT / "encoded_tokens"

with h5py.File(KNN_ROOT / 'mmbody/knn_coverage_analysis_10.h5', 'r') as f:
    A_raw = f['/processed/raw/A'][:]
    idx = f['/coverage_indices/A_uncovered_by_B'][:]
    A_uncovered_raw = A_raw[idx]

    rand_idx = np.random.choice(len(A_uncovered_raw), size=10, replace=False)
    mmbody_uncovered_pose = A_uncovered_raw[rand_idx]

print("Poses in AMASS dataset but not covered by mmbody dataset:")
print(mmbody_uncovered_pose)

with h5py.File(KNN_ROOT / 'mmfi/knn_coverage_analysis_mmfi_128.h5', 'r') as f:
    A_raw = f['/processed/raw/A'][:]
    idx = f['/coverage_indices/A_uncovered_by_B'][:]
    A_uncovered_raw = A_raw[idx]
    rand_idx = np.random.choice(len(A_uncovered_raw), size=10, replace=False)
    mmfi_uncovered_pose = A_uncovered_raw[rand_idx]

print("Poses in AMASS dataset but not covered by MMFi dataset:")
print(mmfi_uncovered_pose)

with h5py.File(TOKEN_ROOT / 'BMLmovi-BMLrub-CMU-GRAB-KIT-MOYO-MoSh-PosePrior-WEIZMANN_tokens.h5', 'r') as f:
    AMASS_tokens = f['/tokens'][:]
    rand_idx = np.random.choice(len(AMASS_tokens), size=10, replace=False)
    AMASS_pose = AMASS_tokens[rand_idx]

print("Tokens in AMASS dataset:")
print(AMASS_pose)

with h5py.File(TOKEN_ROOT / 'MMBody_tokens.h5', 'r') as f:
    mmbody_tokens = f['/tokens'][:]
    rand_idx = np.random.choice(len(mmbody_tokens), size=10, replace=False)
    mmbody_pose = mmbody_tokens[rand_idx]

print("Tokens in MMBody dataset:")
print(mmbody_pose)

with h5py.File(TOKEN_ROOT / 'MMFi_tokens.h5', 'r') as f:
    mmfi_tokens = f['/tokens'][:]
    rand_idx = np.random.choice(len(mmfi_tokens), size=10, replace=False)
    mmfi_pose = mmfi_tokens[rand_idx]

print("Tokens in MMFi dataset:")
print(mmfi_pose)


In [ ]:
# Import necessary modules
import sys
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Add src to path
repo_root = Path("../..").resolve()
os.chdir(repo_root)
sys.path.append(str(repo_root / "src"))

# Import modules
from evaluation.core import load_config, load_model
from evaluation.pose_visualization import plot_multiple_poses, save_pose_plot
print("Successfully imported visualization modules")

# Config and model paths
config_path = repo_root / "src/wicompass/configs" / "joint_vae_base_tokennum16_tokenclass64.json"
model_path = repo_root / "logs/vqvae/vqvae_tokennum16_tokenclass64" / "best_model.pth"

print(f"Config path: {config_path}")
print(f"Model path: {model_path}")
print(f"Config exists: {config_path.exists()}")
print(f"Model exists: {model_path.exists()}")

print("📁 Loading model and config...")
# Load config and model
config = load_config(str(config_path))
model_cfg = config['model']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_model(model_cfg, str(model_path), device)
model.eval()

# Function to convert tokens to poses
def tokens_to_pose(tokens, model, device='cuda', batch_size=32):
    """Convert tokens to pose data"""
    N, token_num = tokens.shape
    model.eval()
    poses_list = []
    
    print(f"🔄 Converting {N} token sequences to poses...")
    
    with torch.no_grad():
        for i in range(0, N, batch_size):
            batch_end = min(i + batch_size, N)
            batch_tokens = tokens[i:batch_end]
            
            # Convert tokens to tensor
            batch_tokens_tensor = torch.from_numpy(batch_tokens).long().to(device)
            
            # Use F.embedding to directly look up vectors in codebook
            quantized_features = torch.nn.functional.embedding(batch_tokens_tensor, model.codebook)
            
            # Use model's decode method to convert to pose
            recovered_joints = model.decode(quantized_features)
            
            # Convert to numpy and add to list
            poses_list.append(recovered_joints.cpu().numpy())
    
    # Concatenate results from all batches
    poses = np.concatenate(poses_list, axis=0)
    print(f"✅ Conversion completed: {poses.shape}")
    return poses

print(f"Model loaded on device: {device}")
print(f"Codebook size: {model.codebook.shape}")

# Convert all tokens to poses for preview
print("\n🎨 Converting tokens to poses...")

# 1. Convert AMASS tokens to poses
print("Converting AMASS tokens...")
AMASS_poses = tokens_to_pose(AMASS_pose, model, device)

# 2. Convert MMBody tokens to poses  
print("Converting MMBody tokens...")
mmbody_poses = tokens_to_pose(mmbody_pose, model, device)

# 3. Convert MMFi tokens to poses
print("Converting MMFi tokens...")
mmfi_poses = tokens_to_pose(mmfi_pose, model, device)

# 4. Analyze uncovered poses data format
print("\n📊 Analyzing uncovered poses data...")
print(f"MMBody uncovered data shape: {mmbody_uncovered_pose.shape}")
print(f"MMFi uncovered data shape: {mmfi_uncovered_pose.shape}")

print("\n✅ All initial conversions completed!")
print(f"AMASS poses: {AMASS_poses.shape}")
print(f"MMBody poses: {mmbody_poses.shape}")
print(f"MMFi poses: {mmfi_poses.shape}")
print(f"MMBody uncovered tokens: {mmbody_uncovered_pose.shape}")
print(f"MMFi uncovered tokens: {mmfi_uncovered_pose.shape}")

In [ ]:
# Quick preview visualization
print("🎨 Creating quick preview visualization...")

# Prepare visualization data - show 3 samples from each dataset
poses_data = []
titles = []

# Add AMASS poses (take first 2)
for i in range(min(2, len(AMASS_poses))):
    poses_data.append((AMASS_poses[i], AMASS_pose[i]))
    titles.append(f"AMASS #{i+1}")

# Add MMBody poses (take first 2)
for i in range(min(2, len(mmbody_poses))):
    poses_data.append((mmbody_poses[i], mmbody_pose[i]))
    titles.append(f"MMBody #{i+1}")

# Add MMFi poses (take first 2)
for i in range(min(2, len(mmfi_poses))):
    poses_data.append((mmfi_poses[i], mmfi_pose[i]))
    titles.append(f"MMFi #{i+1}")

# Create preview visualization
print(f"📊 Creating preview with {len(poses_data)} poses...")
fig = plot_multiple_poses(poses_data, titles, n_cols=3, figsize_per_plot=(4, 4))
fig.suptitle("Quick Preview: AMASS vs MMBody vs MMFi", fontsize=14, fontweight='bold')

# Adjust layout and display
plt.tight_layout()
plt.show()

# Save preview image
output_dir = Path("token_pose_visualization")
output_dir.mkdir(exist_ok=True)
save_pose_plot(fig, output_dir / "datasets_preview.png")
print(f"📁 Preview saved to: {output_dir / 'datasets_preview.png'}")

In [ ]:
# Dataset statistics
print("📊 Dataset Statistics Summary:")
print("=" * 50)

# Statistics for each dataset
datasets_info = [
    ("AMASS", AMASS_pose.shape, AMASS_poses.shape),
    ("MMBody", mmbody_pose.shape, mmbody_poses.shape),
    ("MMFi", mmfi_pose.shape, mmfi_poses.shape),
]

for name, token_shape, pose_shape in datasets_info:
    print(f"{name:12} - Tokens: {token_shape}, Poses: {pose_shape}")

print("\nUncovered Analysis:")
print(f"{'Dataset':12} - {'Uncovered Tokens':20} - {'Description'}")
print("-" * 60)

# Uncovered data statistics from the first cell
print(f"{'MMBody':12} - {str(mmbody_uncovered_pose.shape):20} - AMASS not covered by MMBody")
print(f"{'MMFi':12} - {str(mmfi_uncovered_pose.shape):20} - AMASS not covered by MMFi")

print(f"\n✅ All data loaded and ready for paper visualization generation!")

# Display data loading summary
print("\n🎯 Ready to generate:")
print("  1. MMBody dataset poses (1x10)")
print("  2. MMFi dataset poses (1x10)")  
print("  3. AMASS poses uncovered by MMBody (1x10)")
print("  4. AMASS poses uncovered by MMFi (1x10)")
print("\nAll visualizations will be saved as PDF files without axis labels.")

In [ ]:
# Create 4 separate 1x10 pose visualizations for paper
print("📊 Creating 4 separate 1x10 pose visualizations for paper...")

# First convert uncovered tokens to poses
print("🔄 Converting uncovered tokens to poses...")
mmbody_uncovered_poses = tokens_to_pose(mmbody_uncovered_pose, model, device)
mmfi_uncovered_poses = tokens_to_pose(mmfi_uncovered_pose, model, device)

print("✅ All tokens converted to poses")

# Create output directory
paper_output_dir = Path("paper_pose_visualizations")
paper_output_dir.mkdir(exist_ok=True)

# Import necessary visualization functions
from evaluation.pose_visualization import plot_single_pose

# Define ultra-compact seamless visualization function
def create_ultra_compact_paper_visualization(poses, dataset_name, filename, titles=None):
    """Create ultra-compact seamless 1x10 paper visualization"""
    n_poses = min(10, len(poses))
    
    # Create figure - adjust size ratio
    fig = plt.figure(figsize=(20, 2.2))  # Wider, slightly increased height
    
    # Calculate exact width for each subplot to achieve seamless connection
    total_width = 0.96  # Total available width (leave some margin)
    subplot_width = total_width / n_poses  # Width occupied by each subplot
    start_x = 0.02  # Starting position

    for i in range(n_poses):
        # Calculate exact position for each subplot - no gap
        left = start_x + i * subplot_width
        bottom = 0.05
        width = subplot_width  # Subplot width equals allocated width for seamless connection
        height = 0.9
        
        ax = fig.add_axes([left, bottom, width, height], projection='3d')
        
        # Plot pose
        joints = poses[i]
        title = titles[i] if titles and i < len(titles) else ""
        plot_single_pose(joints, ax, title, None, show_axes=False)
        
        # Remove all margins and spacing
        ax.margins(0, 0, 0)
        
        # Remove axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_zlabel('')
        
        # Hide axis planes
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor('none')
        ax.yaxis.pane.set_edgecolor('none')
        ax.zaxis.pane.set_edgecolor('none')
        
        # Set better 3D viewing angle
        ax.view_init(elev=15, azim=45)
        
        # Set extremely compact axis range
        if joints.shape[0] > 0:
            # Calculate precise boundaries of pose
            x_center = joints[:, 0].mean()
            y_center = joints[:, 1].mean()
            z_center = joints[:, 2].mean()
            
            # Use fixed range for consistency
            range_size = 0.8  # Fixed range size
            
            ax.set_xlim(x_center - range_size/2, x_center + range_size/2)
            ax.set_ylim(y_center - range_size/2, y_center + range_size/2)
            ax.set_zlim(z_center - range_size/2, z_center + range_size/2)
        
        # Set figure boundaries to compact
        ax.set_box_aspect([1,1,1])  # Maintain equal aspect ratio
    
    # Adjust overall figure margins
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
    
    # Save as PDF
    pdf_path = paper_output_dir / f"{filename}.pdf"
    plt.savefig(pdf_path, format='pdf', bbox_inches='tight', dpi=300, 
                pad_inches=0)  # Completely no margins
    print(f"📄 Saved: {pdf_path}")
    
    # Also save PNG version for preview
    png_path = paper_output_dir / f"{filename}.png"
    plt.savefig(png_path, format='png', bbox_inches='tight', dpi=300,
                pad_inches=0)
    
    plt.show()
    plt.close()

# 1. MMBody dataset visualization
print("\n1️⃣ Creating MMBody dataset visualization...")
create_ultra_compact_paper_visualization(
    mmbody_poses, 
    "MMBody", 
    "mmbody_poses_1x10"
)

# 2. MMFi dataset visualization
print("\n2️⃣ Creating MMFi dataset visualization...")
create_ultra_compact_paper_visualization(
    mmfi_poses, 
    "MMFi", 
    "mmfi_poses_1x10"
)

# 3. AMASS poses uncovered by MMBody
print("\n3️⃣ Creating AMASS poses uncovered by MMBody...")
create_ultra_compact_paper_visualization(
    mmbody_uncovered_poses, 
    "AMASS Uncovered by MMBody", 
    "amass_uncovered_by_mmbody_1x10"
)

# 4. AMASS poses uncovered by MMFi
print("\n4️⃣ Creating AMASS poses uncovered by MMFi...")
create_ultra_compact_paper_visualization(
    mmfi_uncovered_poses, 
    "AMASS Uncovered by MMFi", 
    "amass_uncovered_by_mmfi_1x10"
)

print(f"\n✅ All 4 paper visualizations completed!")
print(f"📁 Files saved to: {paper_output_dir}")
print("📄 Generated files:")
for pdf_file in paper_output_dir.glob("*.pdf"):
    print(f"   - {pdf_file.name}")
print("\n🔥 Ultra-compact visualizations with seamless connection between poses!")